# Lab 9: Mini Project, an Object Counter with Alerts

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
You started this course by setting up your Jetson and you end it with a working AI vision application. The program counts one kind of object in the camera picture. When the count reaches your limit it raises an **alert**: it draws the word ALERT, saves a picture, adds a line to a **log** (a CSV file and an Excel file) and sends you an **email** with the picture.

**By the end of this lab you can:**
1. Name the five steps of a vision application (capture, detect, decide, act, record) and find each one in the code.
2. Change the settings so the program matches your own project idea.
3. Tune the confidence limit, the alert limit and the cooldown to balance false alerts and missed alerts.
4. Read the log file in Excel and receive an alert email.
5. Test your project and present it.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Privacy.** The camera sees faces, and the alert pictures are saved and emailed. Only send alerts to your own address, and do not share the pictures.

## 2. Background

**From lab work to an application.** Almost every vision application follows the same five steps. You learned each one in this course:

| Step | What happens | In this notebook | Learned in |
|---|---|---|---|
| Capture | Read a picture from the camera | the live view reads the camera | Lab 6 |
| Detect | Find the objects in it | `model(frame, ...)` in `process` | Labs 7 and 8 |
| Decide | Compare the count with a rule | `count >= alert_at` | Lab 4 |
| Act | Show an alert, save a picture, send an email | `cv2.putText`, `save_picture`, `send_email` | Labs 6 and 9 |
| Record | Write a line in a log file | `log_alert` | Lab 4 |

**Good alerts.** An alert has two ways to fail. A *false alert* fires when nothing is wrong. A *missed alert* does not fire when something is. You tune three settings to balance them:

| Setting | If you raise it | If you lower it |
|---|---|---|
| `CONFIDENCE` | Fewer false alerts, but real objects are missed | More objects found, but more false alerts |
| `ALERT_AT` | Alerts only for bigger groups | Alerts sooner |
| `COOLDOWN` | Fewer pictures and log lines | More pictures and log lines |

The cooldown stops the program from saving a new picture on every frame while the alert is on. Without it the disk and the log would fill in seconds. The email has its own, longer limits (`EMAIL_EVERY` and `EMAIL_MAX`) so your inbox does not fill either.

**Change one thing at a time.** When a program stops working, you must know which change broke it. Change one thing, run it, then change the next. If you break the notebook, close it and type `lab reset 9` in the terminal for a fresh copy.

---
# Part A: Prepare and Plan (20 min)

### Step 1: Get the Model Files
The fast model (`yolo26n_fp16.engine`) is the one you built in Lab 8. If it is missing, the program uses the slower `yolo26n.pt` and still works. This cell copies what exists into this folder.

In [ ]:
import glob
import os
import shutil

print("This notebook is saved in:", os.getcwd())

for source in ["../lab8/yolo26n_fp16.engine", "../lab8/yolo26n.pt", "../models/yolo26n.pt"]:
    name = os.path.basename(source)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
        print("copied", source)

print("Model files here:", sorted(glob.glob("*.engine") + glob.glob("*.pt")))
if not os.path.exists("yolo26n_fp16.engine"):
    print("No FP16 engine here. The program will use yolo26n.pt (slower). Build the engine in Lab 8 to make it faster.")

### Step 2: Plan Your Project
Pick one idea from the table or invent your own. It must use a class from the 80 classes in Lab 7 (for example person, cup, bottle, cell phone, car, cat, dog, bicycle).

| Idea | Class | Alert rule |
|---|---|---|
| Queue counter | person | Alert when 3 or more people are in view |
| Desk tidy check | bottle, cup | Alert when 4 or more items are on the desk |
| Phone reminder | cell phone | Alert when a phone is in view |
| Pet at the door | cat or dog | Alert when one is in view |
| Parking counter | car, bicycle | Alert when 2 or more are in view (use toy cars or a picture) |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Project name:
- What will it count?:
- When will the alert be raised?:
- What will you add or change? (one idea):


---
# Part B: Run the Program (35 min)

### Step 1: The Settings
These are the only lines you must change to get a first working project. Change them to match your plan, then run the cell. Words in capital letters are settings.

To receive alert emails, type **your own email address** in `EMAIL_TO`. Leave it empty if you do not want emails.

In [ ]:
# ---------------- Settings you can change ----------------
TARGET = "person"        # the class to count (any of the 80 class names)
ALERT_AT = 3             # alert when this many or more are in view
CONFIDENCE = 0.40        # confidence limit
COOLDOWN = 5             # seconds between two alerts (picture and log line)
MODEL = "yolo26n_fp16.engine"

EMAIL_TO = ""            # your email address, for example "ali@gmail.com". Empty = no email
EMAIL_EVERY = 60         # at least this many seconds between two emails
EMAIL_MAX = 5            # at most this many emails in one run
# ----------------------------------------------------------
print("Counting", TARGET, "| alert at", ALERT_AT, "| email to:", EMAIL_TO or "(nobody)")

### Step 2: Load the Model
The first time the engine loads, it can take a few seconds. If you see `not found`, the slower `.pt` model is used.

In [ ]:
import csv
import smtplib
import threading
import time
from datetime import datetime
from email.message import EmailMessage

import cv2
import torch
from ultralytics import YOLO

import labview

device = 0 if torch.cuda.is_available() else "cpu"
if not os.path.exists(MODEL):
    print(f"{MODEL} not found, using yolo26n.pt instead (slower)")
    MODEL = "yolo26n.pt"
model = YOLO(MODEL, task="detect")
print("Using model:", MODEL)

### Step 3: The Program
Read this cell before you run it. It has one function for each job:

| Function | What it does |
|---|---|
| `save_picture(img)` | Saves the picture of an alert as `alert_<date>_<time>.jpg` |
| `log_alert(count, picture)` | Adds one line to `alerts.csv` and rewrites `alerts.xlsx` (open it in Excel) |
| `send_email(count, picture)` | Sends an email with the picture in the background, so the video does not freeze |
| `process(frame, state)` | Runs the model on one picture, counts, decides and acts. The live view calls it for every picture. |

In [ ]:
run = {"last_alert": 0.0, "last_email": 0.0, "emails": 0, "note": "", "email": "", "warned": False}


def save_picture(img):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]      # with milliseconds, so two alerts never share a name
    name = f"alert_{stamp}.jpg"
    cv2.imwrite(name, img)
    return name


def write_excel():
    """Rewrite alerts.xlsx from alerts.csv, so it opens in Excel with a header and a number column."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("alerts.xlsx needs openpyxl. alerts.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Alerts"
    with open("alerts.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[2] = int(row[2])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 12, 8, 32)):
        sheet.column_dimensions[column].width = width
    book.save("alerts.xlsx")


def log_alert(count, picture):
    """Add one line to the log files (alerts.csv and alerts.xlsx)."""
    first = not os.path.exists("alerts.csv")
    with open("alerts.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "class", "count", "picture"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), TARGET, count, picture])
    write_excel()


def _deliver(message, account):
    port = int(account.get("PORT", 465))
    if port == 465:
        server = smtplib.SMTP_SSL(account["SERVER"], port, timeout=20)
    else:
        server = smtplib.SMTP(account["SERVER"], port, timeout=20)
        server.starttls()
    server.login(account["USER"], account["PASSWORD"])
    server.send_message(message)
    server.quit()


def make_message(subject, body, picture=None):
    message = EmailMessage()
    message["From"], message["To"], message["Subject"] = mail["USER"], EMAIL_TO, subject
    message.set_content(body)
    if picture:
        with open(picture, "rb") as f:
            message.add_attachment(f.read(), maintype="image", subtype="jpeg", filename=picture)
    return message


def send_email(count, picture):
    """Send the alert email. It runs in a background thread, so the live video keeps going."""
    def job():
        try:
            when = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            _deliver(make_message(f"Myduino alert: {count} {TARGET}",
                                  f"{count} {TARGET} in view at {when}. The picture is attached.", picture), mail)
            run["email"] = f"Email sent to {EMAIL_TO} at {datetime.now().strftime('%H:%M:%S')}"
        except Exception as error:
            run["email"] = f"Email FAILED: {error}"
    threading.Thread(target=job, daemon=True).start()


def maybe_email(count, picture):
    if not EMAIL_TO:
        return
    if mail is None:
        run["email"] = "Email is off: run Step 4 (the test email) first"
        return
    if run["emails"] >= EMAIL_MAX or time.time() - run["last_email"] < EMAIL_EVERY:
        return
    run["last_email"] = time.time()
    run["emails"] += 1
    send_email(count, picture)


def process(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    count = sum(1 for c in result.boxes.cls if result.names[int(c)] == TARGET)
    shown = result.plot()

    alert = count >= int(state["Alert at"])
    colour = (0, 0, 255) if alert else (0, 255, 0)
    cv2.putText(shown, f"{TARGET}: {count}   {state['fps']:.0f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, colour, 2)

    if alert:
        cv2.putText(shown, "ALERT", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        now = time.time()
        if now - run["last_alert"] > COOLDOWN:
            run["last_alert"] = now
            picture = save_picture(shown)
            log_alert(count, picture)
            run["note"] = f"ALERT: {count} {TARGET}, saved {picture}"
            maybe_email(count, picture)

    height = shown.shape[0]
    if run["note"]:
        cv2.putText(shown, run["note"][:70], (10, height - 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    if run["email"]:
        bad = "FAILED" in run["email"] or "off" in run["email"]
        cv2.putText(shown, run["email"][:70], (10, height - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0, 0, 255) if bad else (0, 255, 255), 1)
    return shown


mail = None      # the mail account is set in the next step
print("The program is ready.")

### Step 4: Connect Your Email (skip this if you do not want emails)
The alert emails are sent by a mail account that **your instructor** set up for the class with `lab mail` (you do not need a password). You only type **your own address** in `EMAIL_TO` in Step 1. Run the next cell. It looks for the class account and sends you a **test email** with a picture, so you know it works before the project runs.

If your instructor did not set up an account, the cell asks you for the details of a **spare** mail account. Use an *app password*, never your main password, and do not use an account you care about. This notebook is shown over the classroom network, so everything you type can be seen by others on it.

In [ ]:
import getpass

import numpy as np


def load_mail_account():
    path = os.path.expanduser("~/.lab_mail")
    if os.path.exists(path):
        with open(path) as f:
            return dict(line.rstrip("\n").split("=", 1) for line in f if "=" in line)
    print("No class mail account found. Enter a SPARE account (an app password, not your main password).")
    return {"SERVER": input("SMTP server (for Gmail: smtp.gmail.com): ").strip(),
            "PORT": input("Port (465): ").strip() or "465",
            "USER": input("Sender email address: ").strip(),
            "PASSWORD": getpass.getpass("App password: ")}


def send_test_email():
    global mail
    if not EMAIL_TO:
        print("Type your email address in EMAIL_TO (Step 1), run that cell again, then run this one.")
        return
    mail = load_mail_account()
    test = np.zeros((120, 320, 3), np.uint8)
    cv2.putText(test, "TEST", (80, 80), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
    picture = "test_picture.jpg"
    cv2.imwrite(picture, test)
    try:
        _deliver(make_message("Myduino Jetson: test email",
                              "The email of your Jetson works. Alerts will arrive like this, with a picture.", picture), mail)
        print("Sent. Look in the inbox of", EMAIL_TO, "(and the spam folder).")
    except Exception as error:
        mail = None
        print("The email did not work:", error)
        print("Check the internet, the address and (for Gmail) that the password is an APP password.")


send_test_email()

### Step 5: Run It
Run the cell. The camera picture appears under it with the count and the FPS. Use the sliders to change the confidence limit and the alert limit while it runs. Show your objects to the camera and watch the count. When the count reaches the alert limit, the word ALERT appears, a picture is saved and a line is added to the log. Press **Stop** when you are done.

The first time the engine loads, it can take a few seconds before the picture appears.

In [ ]:
run.update(last_alert=0.0, last_email=0.0, emails=0, note="", email="")
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE),
                                       "Alert at": (1, 10, 1, ALERT_AT)})

### Step 6: See the Results
Run this cell after the alert has fired (and after you pressed Stop). It lists the alert pictures, shows the log and the last picture. The files are in the folder of this notebook: `alerts.csv`, `alerts.xlsx` and the pictures. To open the Excel file, use the file browser on the left of JupyterLab, right-click `alerts.xlsx` and choose **Download**, then open it on your laptop.

In [ ]:
from IPython.display import Image, display


def show_results():
    pictures = sorted(glob.glob("alert_*.jpg"))
    print("Alert pictures:", len(pictures))
    print("Email:", run["email"] or "nothing sent yet")
    for name in ("alerts.csv", "alerts.xlsx"):
        print(f"{name:<12}", "found" if os.path.exists(name) else "not created yet (it appears at the first alert)")
    if os.path.exists("alerts.csv"):
        print()
        with open("alerts.csv") as f:
            print(f.read())
    if pictures:
        display(Image(filename=pictures[-1], width=420))


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model file used (the output of Step 2):
- FPS shown on the screen:
- Did the alert fire when you expected?:
- First line of the log after the header:
- Did the email arrive? How long did it take?:


---
# Part C: Make It Your Own (45 min)
Change the settings first so the program matches your plan from Part A. Then add **at least one improvement** from the list, or one of your own. Change the code in the cells above (for example in `process`), run that cell again, then run Step 5 again.

- Count two classes at the same time, for example cup and bottle, and show both numbers.
- Alert only when the count stays at or above the limit for 3 seconds in a row.
- Draw a rectangle on the picture and count only the objects whose centre is inside it. Use the box corners from Lab 7.
- Show the time of the last alert on the screen.
- Put the count in the email subject, or send a different message for a different count.
- Add a second alert rule, for example a different email when a person and a cell phone are both in view.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell, use *Edit, Copy Cell* before a big change.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Setting 1 you changed and why:
- Setting 2 you changed and why:
- Your improvement and what it does:


---
# Part D: Test and Tune (30 min)
A real project must be tested. Delete the old log first with the cell below. Then try at least **four situations** and record the result.

In [ ]:
# Delete the old log and pictures to start a clean test
for name in glob.glob("alert_*.jpg") + ["alerts.csv", "alerts.xlsx"]:
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("alert*")))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1: normal view with the right objects. Expected / what happened:
- Test 2: objects partly hidden. Expected / what happened:
- Test 3: dim light. Expected / what happened:
- Test 4: a wrong object (not your class). Expected / what happened:


If you get false alerts, raise `CONFIDENCE`. If real objects are missed, lower it. Change it in steps of 0.05 with the slider.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- CONFIDENCE that worked best:
- Number of false alerts in two minutes of testing:
- Number of missed alerts in two minutes of testing:


---
# Part E: Present Your Project (20 min)
Fill in the report. Then give a **3-minute demonstration** to your instructor: show the live view, trigger the alert, show the saved picture, the log and the email, and explain the report.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Project name:
- Class counted and alert rule:
- Model file used and the FPS shown:
- Power mode used (type  sudo nvpmodel -q  in the terminal):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Congratulations.** You started this course by setting up your Jetson and ended with a working AI vision application on a small GPU board. The same steps, prepare the picture, run a model, act on the result and make it fast enough, are the basis of real edge AI products.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The project plan in Part A is complete and the project is clear | |
| 2 | The program runs live with the count and the FPS shown | |
| 3 | The FP16 engine is used (the output of Step 2 does not say `not found`) | |
| 4 | An alert fires, and the saved picture, the log line and the email are shown | |
| 5 | At least one improvement works and you explain the code you changed | |
| 6 | The test tables in Part D are complete and the confidence limit was adjusted for a reason | |
| 7 | The report in Part E is complete | |
| 8 | The 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Name the five steps of a vision application (capture, detect, decide, act, record) and say which function or line in your program does each one.:
- 2. Why does the program use a cooldown? What would happen without it?:
- 3. Your program raises many false alerts. Which setting do you change, and in which direction? What is the cost of this change?:
- 4. Why should you change only one thing at a time, and why keep a copy of a cell before a big change?:
- 5. Explain in words what this line does: `count = sum(1 for c in result.boxes.cls if result.names[int(c)] == TARGET)`:
- 6. Why does the project use the FP16 engine and not yolo26n.pt? What happens if the engine file is missing?:
- 7. A customer wants an alert only when 3 people stay in view for at least 3 seconds. Describe in words how you would change the program.:
- 8. What do `alerts.csv` and the email give you that the screen does not? (a) A faster FPS (b) A permanent record with the time of each alert, and a message when you are not watching (c) Better accuracy (d) Nothing, they are not needed:
- 9. Why must the email be sent in a background thread, and why is there a limit on how many emails are sent?:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `yolo26n_fp16.engine not found, using yolo26n.pt instead` | The engine is not in this folder | Run Step 1 of Part A again, or build the engine in Lab 8 |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose, or the wrong camera number | Press Stop in other live cells, check `lab cameras` in the terminal, restart the kernel (Kernel, Restart Kernel) |
| No picture appears under Step 5 | The cells above were not run, or an earlier error | Run the cells above from the top. Read the error message first. |
| The program never shows ALERT | `TARGET` is spelled wrong, the class is not detected, or `ALERT_AT` is too high | Check the spelling against the Lab 7 class names. Lower the confidence slider. |
| The program shows ALERT all the time | Confidence limit too low, or `ALERT_AT` too small | Raise the confidence slider or `ALERT_AT` |
| `alerts.csv` or `alerts.xlsx` does not exist | No alert has fired yet. The files appear at the first alert. | Make the alert fire, then run Step 6 |
| Many alert pictures are saved | The cooldown is too short | Raise `COOLDOWN` |
| The test email does not arrive | Wrong address, or the message is in the spam folder | Check `EMAIL_TO`, look in spam, run Step 4 again |
| `Authentication failed` or `Username and Password not accepted` | The password is not an app password, or the account blocks sign-in | Ask your instructor to run `lab mail` with an app password |
| `Name or service not known` or `timed out` | The Jetson has no internet | Check the cable and Wi-Fi, ping `8.8.8.8` in the terminal |
| `Email failed:` shown on the picture | The same causes as the test email | Run Step 4 again and read the message |
| `CUDA out of memory` | Other notebooks and programs use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and type `lab reset 9` in the terminal. You get a fresh copy. |

## 7. Further Reading
This was the last lab. To go further, try your own images and models, train a model on a small custom dataset, or add a sensor or a display to your project.
- Ultralytics, Predict mode: https://docs.ultralytics.com/modes/predict/
- Python documentation, csv: https://docs.python.org/3/library/csv.html
- Python documentation, smtplib: https://docs.python.org/3/library/smtplib.html
- openpyxl, working with Excel files: https://openpyxl.readthedocs.io/